# 06 · QAT and deployment artifacts

QAT/TorchAO, ordinary GGUF post-training quantization, and
Unsloth Dynamic GGUF are different experiments. This notebook
keeps their artifacts and acceptance decisions separate.

A custom 1-bit 27B path is research work, not a supported export.
Start with Q4/Q3/Q2 candidates and let long-horizon evaluation decide.

## Install the pinned core environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# The Git pins supply current Unsloth/Qwen3.8 support. Transformers, TRL and
# Datasets deliberately use the mutually compatible versions from the adjacent
# official Unsloth Qwen3.5 27B notebook. Do not replace these with branch-head
# SHAs without resolving package metadata together first.
GIT_REVISIONS = {
    "unsloth": "c87fe20e32aca9ceb2dc5059c2987738f32446e8",
    "unsloth_zoo": "5b239e574f03ab3077c17e49aeef3cacfe7cdd4e",
}

import torch

torch_version = torch.__version__.split("+", 1)[0]
torch_minor = ".".join(torch_version.split(".")[:2])
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
xformers_by_torch = {"2.8": "0.0.32.post2", "2.9": "0.0.33.post1", "2.10": "0.0.34", "2.11": "0.0.34"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed Colab dependency set for torch {torch.__version__}. "
        f"Expected one of {sorted(torchao_by_torch)}; update the compatibility matrix first."
    )

COMPATIBILITY_PINS = {
    "transformers": "5.3.0",
    "trl": "0.22.2",
    "datasets": "4.3.0",
    "peft": "0.19.0",
    "torchao": torchao_by_torch[torch_minor],
    "xformers": xformers_by_torch[torch_minor],
}
INSTALLER_REVISION = "colab-v2"
pin_key = "-".join(value.replace(".", "") for value in COMPATIBILITY_PINS.values())
git_key = "-".join(value[:8] for value in GIT_REVISIONS.values())
INSTALL_KEY = f"{INSTALLER_REVISION}-torch{torch_minor}-{git_key}-{pin_key}"
INSTALL_MARKER = Path(f"/content/.qwen38_env_{INSTALL_KEY}")
PIP_LOG = Path("/content/qwen38_pip_install.log")
FORCE_INSTALL = False

def install_phase(name: str, packages: list[str], *, no_deps: bool = False) -> None:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "--no-cache-dir",
        "--log",
        str(PIP_LOG),
    ]
    if no_deps:
        command.append("--no-deps")
    command.extend(packages)
    print(f"\n=== install phase: {name} ===")
    print("\n".join(f"  {package}" for package in packages))
    result = subprocess.run(command, check=False)
    if result.returncode:
        log_tail = (
            "\n".join(PIP_LOG.read_text(errors="replace").splitlines()[-120:])
            if PIP_LOG.exists()
            else "[pip did not create its log file]"
        )
        print(f"\n--- tail of {PIP_LOG} ---\n{log_tail}")
        raise RuntimeError(
            f"Package installation failed during {name!r} with exit code {result.returncode}. "
            f"The detailed log is at {PIP_LOG}."
        )

if FORCE_INSTALL or not INSTALL_MARKER.exists():
    if PIP_LOG.exists():
        PIP_LOG.unlink()
    install_phase("packaging tools", ["pip", "setuptools==80.9.0", "wheel>=0.42.0"])
    install_phase("Qwen3.8 training stack", [
        f"unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git@{GIT_REVISIONS['unsloth_zoo']}",
        f"unsloth @ git+https://github.com/unslothai/unsloth.git@{GIT_REVISIONS['unsloth']}",
        f"torch=={torch_version}",
        f"torchao=={COMPATIBILITY_PINS['torchao']}",
        f"transformers=={COMPATIBILITY_PINS['transformers']}",
        f"trl=={COMPATIBILITY_PINS['trl']}",
        f"datasets=={COMPATIBILITY_PINS['datasets']}",
        f"peft=={COMPATIBILITY_PINS['peft']}",
        "accelerate",
        "bitsandbytes",
        "trackio",
        "huggingface_hub>=0.34.0,<2.0",
        "hf_transfer",
        "sentencepiece>=0.2.0",
        "protobuf",
        "pytest",
        "jmespath",
    ])
    install_phase(
        "PyTorch-matched xFormers wheel",
        [f"xformers=={COMPATIBILITY_PINS['xformers']}"],
        no_deps=True,
    )
    INSTALL_MARKER.write_text(INSTALL_KEY)
    print("Packages installed. Restart the Colab runtime, then rerun this notebook from the top.")
else:
    print(f"Pinned environment already installed: {INSTALL_KEY}")

After the first install, restart the runtime and rerun the notebook from the top; the install marker skips the pip work.

In [ ]:
import json
import os
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import torch
from huggingface_hub import login, whoami

if "GIT_REVISIONS" not in globals():
    raise RuntimeError(
        "This runtime was restarted. Rerun the notebook from the first cell; "
        "the install marker will skip the expensive package installation."
    )
if "COMPATIBILITY_PINS" not in globals():
    raise RuntimeError("Missing compatibility pins; rerun the notebook from the first cell.")

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab G4 GPU runtime before continuing.")

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
# A vendor-labelled 96 GB card can be reported as about 89.4 GiB because
# PyTorch converts the byte count with a binary divisor. Keep the floor well
# above the roughly 44.7 GiB reported for a 48 GB card without rejecting G4.
MIN_G4_TOTAL_GIB = 85.0
print(
    f"GPU: {gpu.name} ({gpu_total_gib:.1f} GiB total), "
    f"capability={torch.cuda.get_device_capability(0)}"
)
if gpu_total_gib < MIN_G4_TOTAL_GIB:
    raise RuntimeError(
        "This suite expects the nominal 96 GB Colab G4 runtime. "
        f"PyTorch reports {gpu_total_gib:.1f} GiB total; expected at least "
        f"{MIN_G4_TOTAL_GIB:.0f} GiB. A value near 45 GiB usually indicates "
        "the 48 GB GPU variant."
    )

hf_token = userdata.get("HF_TOKEN") if userdata is not None else os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN to Colab Secrets before continuing.")
login(token=hf_token, add_to_git_credential=False)
HF_USERNAME = whoami()["name"]

def package_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"

observed_pins = {name: package_version(name) for name in COMPATIBILITY_PINS}
pin_mismatches = {
    name: {"expected": expected, "observed": observed_pins[name]}
    for name, expected in COMPATIBILITY_PINS.items()
    if observed_pins[name] != expected
}
if pin_mismatches:
    raise RuntimeError(
        "The runtime does not match the reviewed compatibility set. "
        f"Rerun the install cell with FORCE_INSTALL=True: {pin_mismatches}"
    )

RUN_ROOT = Path("/content/qwen38_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
runtime_manifest = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "packages": {
        name: package_version(name)
        for name in ["unsloth", "unsloth_zoo", "transformers", "trl", "peft", "datasets"]
    },
    "git_revisions": GIT_REVISIONS,
    "compatibility_pins": COMPATIBILITY_PINS,
}
(RUN_ROOT / "runtime_manifest.json").write_text(json.dumps(runtime_manifest, indent=2))
print(json.dumps(runtime_manifest, indent=2))
print(f"Authenticated as {HF_USERNAME}")

## Install TorchAO/Fbgemm versions matched to Colab PyTorch

In [ ]:
import re

torch_minor = re.match(r"\d+\.\d+", torch.__version__).group(0)
torchao_by_torch = {"2.8": "0.16.0", "2.9": "0.16.0", "2.10": "0.16.0", "2.11": "0.18.0"}
fbgemm_by_torch = {"2.8": "1.3.0", "2.9": "1.4.2", "2.10": "1.5.0", "2.11": "1.5.0"}
if torch_minor not in torchao_by_torch:
    raise RuntimeError(
        f"No reviewed TorchAO pin for torch {torch_minor}; update the mapping from the Unsloth notebook catalog."
    )
INSTALL_QAT_DEPS = False
if INSTALL_QAT_DEPS:
    import numpy as np

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall",
        f"torchao=={torchao_by_torch[torch_minor]}",
        f"fbgemm-gpu-genai=={fbgemm_by_torch[torch_minor]}",
        f"numpy=={np.__version__}",
    ])
    print("Restart the runtime, rerun setup, then leave INSTALL_QAT_DEPS=False.")
else:
    print({"torch": torch.__version__, "planned_torchao": torchao_by_torch[torch_minor], "planned_fbgemm": fbgemm_by_torch[torch_minor]})

## Artifact configuration

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset, load_dataset
from trl import SFTConfig, SFTTrainer

ACCEPTED_ADAPTER_ID = f"{HF_USERNAME}/qwen38-27b-code-sft-lora"
ACCEPTED_REVISION = "REPLACE_WITH_ACCEPTED_COMMIT"
MERGED_MODEL_ID = f"{HF_USERNAME}/qwen38-27b-code-accepted-merged"
QAT_OUTPUT_ID = f"{HF_USERNAME}/qwen38-27b-code-qat-int4"
GGUF_OUTPUT_ID = f"{HF_USERNAME}/qwen38-27b-code-gguf"
DATASET_ID = f"{HF_USERNAME}/qwen38-code-native-sft-v0"
DATASET_REVISION = "REPLACE_WITH_DATASET_COMMIT"
MAX_SEQ_LENGTH = 4_096

RUN_QAT = False
PUSH_QAT = False
RUN_STANDARD_GGUF_EXPORT = False
BUILD_CALIBRATION_CORPUS = False

if any([RUN_QAT, RUN_STANDARD_GGUF_EXPORT, BUILD_CALIBRATION_CORPUS]):
    if ACCEPTED_REVISION.startswith("REPLACE_"):
        raise RuntimeError("Pin the accepted adapter revision before export.")

## QAT-LoRA branch (fresh adapter from an accepted merged checkpoint)

In [ ]:
if RUN_QAT:
    try:
        import torchao
        from torchao.quantization import quantize_
        from torchao.quantization.qat import QATConfig
    except ImportError as exc:
        raise RuntimeError("Install the matched TorchAO/Fbgemm pair and restart first.") from exc

    qat_model, qat_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MERGED_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=False,
        token=hf_token,
    )
    qat_model = FastLanguageModel.get_peft_model(
        qat_model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj", "in_proj", "out_proj",
        ],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        qat_scheme="int4",
    )
    for name, parameter in qat_model.named_parameters():
        if any(part in name.lower() for part in ("vision", "visual", "image")):
            parameter.requires_grad_(False)
    fake_quant_modules = [
        module.__class__.__name__ for module in qat_model.modules()
        if "FakeQuantized" in module.__class__.__name__
    ]
    if not fake_quant_modules:
        raise RuntimeError("qat_scheme did not install fake-quantized modules; stop before training.")
    print({"fake_quantized_modules": len(fake_quant_modules)})

    qat_data = load_dataset(
        DATASET_ID,
        split="train",
        revision=DATASET_REVISION,
        token=hf_token,
    )
    if "text" not in qat_data.column_names:
        raise RuntimeError("Publish the rendered `text` field from notebook 02 before QAT.")
    qat_args = SFTConfig(
        output_dir=str(RUN_ROOT / "qat"),
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        max_steps=100,
        bf16=True,
        optim="adamw_8bit",
        logging_steps=1,
        save_steps=25,
        report_to="trackio",
        run_name="qwen38-code-qat-int4",
    )
    qat_trainer = SFTTrainer(
        model=qat_model,
        processing_class=qat_tokenizer,
        train_dataset=qat_data,
        args=qat_args,
    )
    qat_trainer = train_on_responses_only(
        qat_trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    qat_labels = next(iter(qat_trainer.get_train_dataloader()))["labels"]
    assert (qat_labels != -100).any(), "QAT response masking removed every target token."
    qat_trainer.train()

    # Convert the fake-quantized representation only after training.
    quantize_(qat_model, QATConfig(step="convert"))
    qat_dir = RUN_ROOT / "qat" / "torchao_int4"
    qat_model.save_pretrained_torchao(
        str(qat_dir),
        qat_tokenizer,
    )
    qat_tokenizer.save_pretrained(str(qat_dir))
    if PUSH_QAT:
        from huggingface_hub import HfApi
        HfApi(token=hf_token).upload_folder(
            repo_id=QAT_OUTPUT_ID,
            folder_path=str(qat_dir),
            repo_type="model",
        )
else:
    print("QAT is disabled. It requires an accepted merged source and matched TorchAO installation.")

## Standard GGUF control artifacts

In [ ]:
if RUN_STANDARD_GGUF_EXPORT:
    export_model, export_tokenizer = FastLanguageModel.from_pretrained(
        model_name=ACCEPTED_ADAPTER_ID,
        revision=ACCEPTED_REVISION,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=False,
        token=hf_token,
    )
    export_model.push_to_hub_gguf(
        GGUF_OUTPUT_ID,
        export_tokenizer,
        quantization_method=["q8_0", "q5_k_m", "q4_k_m"],
        token=hf_token,
    )
    print("Published standard llama.cpp GGUF controls. These are not Unsloth Dynamic quants.")
else:
    print("Standard GGUF export is disabled; it can take substantial disk, RAM, and upload time.")

## Build a native-template calibration corpus for later low-bit conversion

In [ ]:
if BUILD_CALIBRATION_CORPUS:
    from transformers import AutoTokenizer

    calibration_tokenizer = AutoTokenizer.from_pretrained(
        ACCEPTED_ADAPTER_ID,
        revision=ACCEPTED_REVISION,
        token=hf_token,
    )
    calibration = load_dataset(
        DATASET_ID,
        split="train",
        revision=DATASET_REVISION,
        token=hf_token,
    )
    texts = calibration["text"][:512]
    token_lengths = [
        len(calibration_tokenizer(text=text, add_special_tokens=False)["input_ids"])
        for text in texts
    ]
    calibration_path = RUN_ROOT / "calibration_native_tools.txt"
    calibration_path.write_text("\n<|calibration_document|>\n".join(texts))
    print({
        "path": str(calibration_path),
        "documents": len(texts),
        "tokens": sum(token_lengths),
        "max_tokens": max(token_lengths),
    })
else:
    print("Calibration build is disabled.")

## Dynamic Q3/Q2 and the 1-bit boundary

Do not relabel the standard exports above as Dynamic GGUF. First
evaluate the published Unsloth Dynamic Q4/Q3/Q2 artifacts, if
available for the accepted checkpoint. A custom importance-matrix
conversion must pin the exact `llama.cpp` and Unsloth converter
revisions and use the native-template calibration corpus.

There is no supported one-bit Qwen3.8-27B path in this suite.
Treat it as a separate research branch only after Q2 fails the
frozen long-horizon gate. Quantization acceptance is based on
complete episode success, not perplexity alone.